# Mini2P single-run QC: bidiphase, destriping, distortion, Suite2p registration

This notebook inspects one recording. All processing lives in the `mini2p_preprocessing` package; the cells here only call it and draw figures, so what you see is exactly what the batch runs produce.

Order of operations:

1. Correct the odd/even-line scan phase of both channels. With `phase.method = "per_frame"` (the default) every frame gets its own phase: from the structural channel wherever its per-frame phase is consistent and agrees with the functional channel, otherwise from the functional channel over `phase.fallback_window_frames`. With `"chunked"`, one phase per chunk of `phase.chunk_frames` timepoints (optionally a second residual pass).
2. Optionally destripe (`destripe.enabled`), preserving each frame's mean fluorescence.
3. Distortion-correct with the fixed calibration (`distortion.mode`).
4. Register with Suite2p, with Suite2p's own bidiphase correction disabled.

Outputs go to `<run_dir>/without_destriping` or `<run_dir>/with_destriping` (inside `<run_dir>/without_distortion_correction/` when `[distortion] mode = "off"`), so to compare the two, run the notebook once with `[destripe] enabled = false` and once with `true`. Each variant gets its own TIFFs, sidecars and `suite2p/` folder.

Settings and the recording to inspect come from `config/pipeline.toml` (copy `config/pipeline.example.toml`). Change settings there, not here, so the notebook and batch runs always agree.

In [ ]:
import json
import logging
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

from mini2p_preprocessing import plots, qc
from mini2p_preprocessing.bidiphase import adjacent_line_score, apply_bidiphase, select_alignment
from mini2p_preprocessing.config import (
    corrected_tiff_path,
    load_qc_run,
    load_settings,
    distortion_corrected,
    run_name_from_output_dir,
    variant_output_dir,
)
from mini2p_preprocessing.contrast import scale_contrast
from mini2p_preprocessing.correction import correction_stages, preprocess_tiff
from mini2p_preprocessing.destripe import detrend_chunk
from mini2p_preprocessing.distortion import write_distortion_corrected
from mini2p_preprocessing.exports import Contrast, save_contrast_choice, with_saved_contrast
from mini2p_preprocessing.extraction import (
    EXTRACTION_RECORD,
    load_cellpose_labels,
    run_cellpose_extraction,
)
from mini2p_preprocessing.frame_phase import apply_frame_phases, fallback_stretches
from mini2p_preprocessing.fluorescence import (
    audit_table,
    compute_dff,
    green_red_dff,
    select_frames,
)
from mini2p_preprocessing.pipeline import suite2p_input_for
from mini2p_preprocessing.plane import load_plane_traces, load_valid_frames
from mini2p_preprocessing.registration import load_registered_outputs, run_registration

logging.basicConfig(level=logging.INFO, format="%(name)s: %(message)s", force=True)

CONFIG = Path("../config/pipeline.toml")
settings = load_settings(CONFIG)
run = load_qc_run(CONFIG)
OUTPUT_DIR = variant_output_dir(
    run.run_dir, settings.destripe.enabled, distortion_corrected=distortion_corrected(settings)
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CORRECTED_TIFF = corrected_tiff_path(OUTPUT_DIR, run.input_tiff, settings)
CHANNEL_NAMES = settings.channels.names

print("Input:", run.input_tiff)
print("Output folder:", OUTPUT_DIR)
print("Destriping:", settings.destripe.enabled)
print("Phase method:", settings.phase.method, "| chunked phase source:", settings.phase.alignment_source,
      "| motion source:", settings.registration.motion_alignment_source)

## Preview phase estimates and search quality

Run this before writing the full TIFF.

**`method = "per_frame"`** (default): measures every frame of both channels (about a minute and a half per 9,000 frames) and plots each frame's phase in each channel, the phase that will be applied (black), grey where the functional channel set it, and the structural line correlation lost at the applied phase. The measurement is reused when the TIFF is written below.

- Long grey stretches: the structural channel had no usable tissue there, so those frames rely on the functional channel's `fallback_window_frames` average, which smooths over fast jumps.
- Spikes in the bottom panel: frames the correction did not fully fix.
- Values reaching ±`max_shift_px`: increase the search range.

**`method = "chunked"`**: top, phase per chunk (ringed points are at or near the search boundary); bottom, best normalized adjacent-line score per channel. Estimates jumping within a section whose mean stays doubled: reduce `chunk_frames`. The score curve for the lowest-scoring chunk reveals ambiguous competing peaks.

In [ ]:
PER_FRAME = settings.phase.method == "per_frame"
if PER_FRAME:
    phase_preview = qc.frame_phase_preview(run.input_tiff, settings)
    plots.plot_frame_phases(phase_preview.chosen)
    plt.show()
    chosen = phase_preview.chosen
    print("Phase range:", (int(chosen.phase_px.min()), int(chosen.phase_px.max())), "px")
    print(f"Structural channel used for {100 * chosen.used_structural.mean():.0f}% of frames")
    stretches = fallback_stretches(chosen.used_structural)
    print(f"Functional channel used in {len(stretches)} stretch(es):", stretches[:20],
          "..." if len(stretches) > 20 else "")
    if max(abs(int(chosen.phase_px.min())), abs(int(chosen.phase_px.max()))) >= settings.phase.max_shift_px:
        print("WARNING: phases reach the search boundary - increase phase.max_shift_px and rerun.")
else:
    trace = qc.phase_trace(run.input_tiff, settings)
    plots.plot_phase_trace(trace, settings)
    plt.show()

    phases = [row["phase_px"] for row in trace]
    print("Phase range:", (min(phases), max(phases)))
    boundary_starts = [row["start"] for row in trace if row["at_boundary"]]
    if boundary_starts:
        print("WARNING: search-boundary chunks start at", boundary_starts,
              "- increase phase.max_shift_px and rerun.")

    worst = qc.lowest_score_chunk(trace)
    worst_chunks = qc.load_chunk(run.input_tiff, worst["start"], worst["end"], settings.channels.n_channels)
    if settings.phase.estimator == "wide":
        worst_choice = select_alignment(worst_chunks, settings.phase, settings.channels)
        plots.plot_score_curve(
            worst_choice.estimate,
            f"Lowest-score chunk {worst['start']}:{worst['end']} ({worst_choice.selected_source})",
        )
        plt.show()

## Manual offset for the lowest-scoring chunk (`method = "chunked"` only)

The slider applies an integer offset to both channels of the chunk above and compares it with the uncorrected mean. Edge columns affected by the largest shift are hidden. When it looks right, copy the printed line into `[phase.manual_first_phase_by_start]` in the config. Nothing is written here. The per-frame method has no manual overrides; its plot above shows every frame instead.

In [ ]:
if PER_FRAME:
    print('Manual chunk offsets apply to phase.method = "chunked" only.')
else:
    raw_means = qc.mean_images(worst_chunks)
    crop = settings.phase.max_shift_px
    x_slice = slice(crop, -crop) if 2 * crop < raw_means[0].shape[1] else slice(None)
    score_channel = worst["selected_channel_zero_based"]


    def show_offset(offset_px):
        corrected_means = qc.mean_images([apply_bidiphase(c, offset_px) for c in worst_chunks])
        axes = plots.plot_channel_pairs(
            raw_means, corrected_means, ("Uncorrected", f"Manual offset {offset_px:+d} px"),
            CHANNEL_NAMES, x_slice=x_slice,
        )
        score = adjacent_line_score(corrected_means[score_channel], crop)
        axes[0, 0].figure.suptitle(
            f"Chunk {worst['start']}:{worst['end']} — score {score:.4f} "
            f"(automatic {worst['phase_px']:+d} px)", fontsize=14,
        )
        plt.show()
        print(f"Config line under [phase.manual_first_phase_by_start]:  {worst['start']} = {offset_px}")


    slider = widgets.IntSlider(value=worst["phase_px"], min=-crop, max=crop, step=1,
                               description="Offset (px):", continuous_update=False,
                               layout=widgets.Layout(width="650px"))
    display(slider, widgets.interactive_output(show_offset, {"offset_px": slider}))

## Write the corrected TIFF

Writes `<stem>_dynamic_bidiphase[_destriped].tif` plus a `.phase.json` sidecar with the settings. With `method = "per_frame"` it also writes `.phase_frames.npz` (every frame's phase in each channel, which channel was used, consistency, correlation lost, and the measured curves); with `"chunked"` the JSON records every chunk's phases. With `overwrite = false` an existing file is never replaced.

In [ ]:
correction = preprocess_tiff(
    run.input_tiff, CORRECTED_TIFF, settings, run_name_from_output_dir(OUTPUT_DIR),
    line_curves=phase_preview.curves if PER_FRAME else None,  # reuse the preview's measurement
)
if not PER_FRAME:
    print("Whole-recording second phase:", correction.global_second_phase_px)

## QC 1, 1b, 2 — each correction stage for one stretch

Uses exactly the phases that were written, so these are the written pixels (for `"chunked"`, when `qc_start_timepoint` is a multiple of `phase.chunk_frames`). The stretch is `phase.chunk_frames` timepoints long. Each row is one channel; both columns share display limits.

In [ ]:
qc_start = run.qc_start_timepoint
qc_raw = qc.load_chunk(run.input_tiff, qc_start, qc_start + settings.phase.chunk_frames,
                       settings.channels.n_channels)
if PER_FRAME:
    qc_phases = correction.frame_phases.phase_px[qc_start:qc_start + len(qc_raw[0])]
    qc_used = correction.frame_phases.used_structural[qc_start:qc_start + len(qc_raw[0])]
    after_phase = [apply_frame_phases(c, qc_phases) for c in qc_raw]
    print(f"Timepoints {qc_start}:{qc_start + len(qc_raw[0])}; phases {qc_phases.min():+d}..{qc_phases.max():+d} px; "
          f"structural used for {100 * qc_used.mean():.0f}% of these frames")
    plots.plot_channel_pairs(qc.mean_images(qc_raw), qc.mean_images(after_phase),
                             ("Before", "After per-frame bidiphase"), CHANNEL_NAMES)
    plt.show()
    if settings.destripe.enabled:
        destriped = [detrend_chunk(c, settings.destripe.preserve_frame_mean) for c in after_phase]
        plots.plot_channel_pairs(qc.mean_images(after_phase), qc.mean_images(destriped),
                                 ("Before destriping", "After destriping"), CHANNEL_NAMES)
        plt.show()
else:
    stages = correction_stages(qc_raw, qc_start, settings, correction.global_residual)
    print(f"Timepoints {qc_start}:{qc_start + len(qc_raw[0])}; first={stages.first_phase_px:+d} px; "
          f"second={stages.second_phase_px:+d} px; source={stages.first_choice.selected_source}; "
          f"scores={stages.first_choice.score_by_source}")

    plots.plot_channel_pairs(qc.mean_images(qc_raw), qc.mean_images(stages.after_first),
                             ("Before", "After first bidiphase"), CHANNEL_NAMES)
    plt.show()
    if settings.phase.run_second_pass:
        plots.plot_channel_pairs(qc.mean_images(stages.after_first), qc.mean_images(stages.after_second),
                                 ("After first bidiphase", f"After second ({settings.phase.second_pass_mode})"),
                                 CHANNEL_NAMES)
        plt.show()
    if settings.destripe.enabled:
        plots.plot_channel_pairs(qc.mean_images(stages.after_second), qc.mean_images(stages.corrected),
                                 ("Before destriping", "After destriping"), CHANNEL_NAMES)
        plt.show()

## Distortion correction (before registration)

`local_batched` applies one fixed calibration transform to every frame, `distortion.batch_frames` timepoints at a time; the transform and crop are computed once for the recording, so batch size never changes the pixels. `cluster_full` expects the SLURM worker's output to exist already; `off` registers the corrected TIFF directly.

In [ ]:
SUITE2P_INPUT = suite2p_input_for(OUTPUT_DIR, run.input_tiff, settings)
if settings.distortion.mode == "local_batched":
    write_distortion_corrected(CORRECTED_TIFF, SUITE2P_INPUT, settings)
elif settings.distortion.mode == "cluster_full" and not SUITE2P_INPUT.is_file():
    raise FileNotFoundError(f"Cluster distortion output not found: {SUITE2P_INPUT}")
print("Suite2p input:", SUITE2P_INPUT)

if settings.distortion.mode != "off":
    n = run.qc_distortion_frames
    before = qc.mean_images(qc.load_chunk(CORRECTED_TIFF, 0, n, settings.channels.n_channels))
    after = qc.mean_images(qc.load_chunk(SUITE2P_INPUT, 0, n, settings.channels.n_channels))
    plots.plot_channel_pairs(before, after, ("Before distortion correction", "After"), CHANNEL_NAMES)
    plt.show()
    print("Distortion output shape:", after[0].shape)

## Suite2p registration

Suite2p's `do_bidiphase` is off because the scan phase was corrected above. `registration.motion_alignment_source` picks the channel that supplies motion shifts; they are applied to both channels. Green stays the functional channel.

In [ ]:
PLANE_DIR = run_registration(SUITE2P_INPUT, OUTPUT_DIR, settings)
registered = load_registered_outputs(PLANE_DIR)
print("Suite2p output:", PLANE_DIR)

## QC 4 and registration diagnostics

Before/after means over the same first frames; then rigid offsets, frame-to-reference correlation, bad frames, nonrigid offsets, and the first-N registered mean against Suite2p's recording-wide mean. Large offset jumps at phase-chunk boundaries suggest reducing `chunk_frames`.

In [ ]:
n = run.qc_registration_frames
before_registration = qc.mean_images(qc.load_chunk(SUITE2P_INPUT, 0, n, settings.channels.n_channels))
first_n_registered = qc.registered_first_n_means(registered, n, settings)
plots.plot_channel_pairs(before_registration, first_n_registered,
                         ("Before Suite2p registration", "After"), CHANNEL_NAMES)
plt.show()

ops = registered.ops
for key in ["fs", "nframes", "Ly", "Lx", "nchannels", "functional_chan", "align_by_chan2",
            "do_bidiphase", "bidiphase", "nonrigid", "block_size", "maxregshift", "maxregshiftNR",
            "smooth_sigma", "smooth_sigma_time", "nimg_init", "batch_size",
            "two_step_registration", "norm_frames", "th_badframes", "snr_thresh"]:
    if key in ops:
        print(f"{key:>24}: {ops[key]}")

plots.plot_registration_traces(ops, settings.registration.fs_hz)
plt.show()
print(f"Bad frames: {int(np.sum(ops.get('badframes', [])))}/{len(ops.get('xoff', []))}")
if np.ndim(ops.get("xoff1", [])) >= 2:
    plots.plot_nonrigid_offsets(ops, settings.registration.fs_hz)
    plt.show()

functional_mean, structural_mean = qc.suite2p_mean_images(ops)
by_channel = {settings.channels.functional_channel: functional_mean,
              settings.channels.structural_channel: structural_mean}
plots.plot_channel_pairs(first_n_registered,
                         [by_channel[ch] for ch in range(1, settings.channels.n_channels + 1)],
                         (f"First {n} registered frames", "Suite2p recording-wide mean"),
                         CHANNEL_NAMES)
plt.show()

## Registered mean images: contrast and overlay

Contrast controls affect display only. The next cell saves your chosen contrast beside `ops.npy`.

In [ ]:
def contrast_sliders(prefix):
    return (
        widgets.FloatSlider(value=1.0, min=0.0, max=50.0, step=0.5, description=f"{prefix} black %",
                            continuous_update=False),
        widgets.FloatSlider(value=99.8, min=90.0, max=100.0, step=0.05,
                            description=f"{prefix} white %", continuous_update=False),
        widgets.FloatSlider(value=1.0, min=0.2, max=3.0, step=0.05, description=f"{prefix} gamma",
                            continuous_update=False),
    )


green_sliders, red_sliders = contrast_sliders("Green"), contrast_sliders("Red")


def preview(gb, gw, gg, rb, rw, rg):
    if gw <= gb or rw <= rb:
        print("Each white percentile must be above its black percentile")
        return
    green, *green_range = scale_contrast(functional_mean, gb, gw, gg)
    red, *red_range = scale_contrast(structural_mean, rb, rw, rg)
    plots.plot_contrast_panels(green, red, green_range, red_range)
    plt.show()


keys = ["gb", "gw", "gg", "rb", "rw", "rg"]
display(widgets.HBox([widgets.VBox(green_sliders), widgets.VBox(red_sliders)]),
        widgets.interactive_output(preview, dict(zip(keys, (*green_sliders, *red_sliders)))))

In [ ]:
chosen = [Contrast(*(s.value for s in sliders)) for sliders in (green_sliders, red_sliders)]
for label, path in save_contrast_choice(PLANE_DIR, functional_mean, structural_mean, *chosen).items():
    print(f"Saved {label}: {path}")

## Fluorescence audit: F, F0 and ΔF/F0

Workflow: register above, segment the registered mean image in Cellpose (`meanImg_green_contrast.png` from the contrast cell above, or the batch export's `meanImg_green_functional_contrast.png`), then extract and audit here.

- **Extraction** builds Suite2p `stat` from the Cellpose masks and extracts both channels with identical masks. It **replaces** `F`, `Fneu`, `stat`, `iscell` and `spks` (and `F_chan2`, `Fneu_chan2`) in this run's `suite2p/plane0`. It is off unless you set `RUN_CELLPOSE_EXTRACTION = True`.
- **Masks from another registration.** If the masks were drawn on a mean image from a different Suite2p run of the same recording (for example the first 1000 frames instead of the full session), set `[qc] cellpose_image` to that image. Extraction then measures the offset to this plane's mean image, shifts the masks onto it, and records the shift. It refuses if the offset is not the same across the image, because a single shift cannot fix that.
- **ΔF/F0.** Suite2p's `preprocess` returns ΔF = Fcorr − F0, not ΔF/F0. Here F0 is computed explicitly from `Fcorr = F − c·Fneu`, and ΔF/F0 is formed as ΔF divided by F0. It is left as NaN wherever F0 ≤ `fluorescence.min_f0`, where the ratio has no meaning.
- With `[qc] valid_frames` set, frames are selected before the baseline is computed, as in the analysis code, so the baseline window spans any gaps and the time axis counts valid frames only.

In [ ]:
PLANE_DIR = OUTPUT_DIR / settings.registration.save_folder / "plane0"
RUN_CELLPOSE_EXTRACTION = False  # True replaces F, Fneu, stat, iscell and spks in PLANE_DIR
if RUN_CELLPOSE_EXTRACTION:
    if run.cellpose_seg is None:
        raise ValueError("Set [qc] cellpose_seg in the config to this run's Cellpose *_seg.npy")
    run_cellpose_extraction(run.cellpose_seg, PLANE_DIR, settings, mask_image=run.cellpose_image)

record = {}
record_file = PLANE_DIR / EXTRACTION_RECORD
if record_file.is_file():
    record = json.loads(record_file.read_text())
    alignment = record.get("mask_alignment")
    print("Masks:", record["seg_file"])
    if alignment:
        print(f"Shifted by dy={alignment['dy']}, dx={alignment['dx']} px to fit this plane; "
              f"quadrants {alignment['quadrant_shifts']}; rank correlation "
              f"{alignment['rank_correlation_before']:.3f} -> {alignment['rank_correlation_after']:.3f}")
        if alignment["dropped_labels"]:
            print("Masks dropped (mostly pushed off the image):", alignment["dropped_labels"])
    else:
        print("Masks used as segmented (no alignment)")

In [ ]:
traces = load_plane_traces(PLANE_DIR)
frames = load_valid_frames(run.valid_frames, run.valid_frames_key) if run.valid_frames else None
f, fneu = select_frames(traces.f, frames), select_frames(traces.fneu, frames)
fl = settings.fluorescence
green = compute_dff(f, fneu, traces.fs, fl)
table = audit_table(f, fneu, green)
ratio = None
if traces.f_chan2 is not None:
    ratio = green_red_dff(f, fneu, select_frames(traces.f_chan2, frames),
                          select_frames(traces.fneu_chan2, frames), traces.fs, fl)

print(f"{f.shape[0]} ROIs x {f.shape[1]} frames "
      f"({'valid frames only' if frames is not None else 'all frames'}) at {traces.fs:g} Hz")
print(f"Neuropil coefficient {fl.neuropil_coefficient}; {fl.baseline} baseline; "
      f"ΔF/F0 is NaN where F0 <= {fl.min_f0:g}")
invalid_rois = np.flatnonzero(table["frac_invalid_f0"] > 0)
negative_rois = np.flatnonzero(table["frac_negative_fcorr"] > 0.1)
print(f"ROIs with an invalid F0 somewhere: {len(invalid_rois)} {invalid_rois[:30].tolist()}")
print(f"ROIs with Fcorr < 0 in >10% of frames: {len(negative_rois)} {negative_rois[:30].tolist()}")
plots.plot_f0_overview(table, fl.min_f0)
plt.show()

In [ ]:
# The green mean image with the contrast saved above, i.e. as in meanImg_green_contrast.png.
# Used by the mask and ROI views below. Save the contrast first if this fails.
mean_img = with_saved_contrast(PLANE_DIR, traces.ops["meanImg"], "green")


In [ ]:
# The masks actually used, from the extraction record (not the current config).
if record.get("seg_file"):
    shift = record.get("mask_alignment") or {"dy": 0, "dx": 0}
    plots.plot_mask_alignment(mean_img,
                              load_cellpose_labels(record["seg_file"]), traces.labels,
                              title=f"dy={shift['dy']}, dx={shift['dx']}")
    plt.show()
else:
    print("No Cellpose extraction recorded for this plane; showing no mask overlay")

### One ROI at a time

Top to bottom: raw F and neuropil; corrected F with its F0 (shaded where F0 is invalid); ΔF/F0; and, with two channels, the green/red ratio Δ(G/R)/(G/R)0. On the right, the ROI outlined in red on the Suite2p mean image (with the contrast saved above), with the other ROIs in blue.

In [ ]:
time_label = "Valid-frame time (s)" if frames is not None else "Time (s)"
p_cell = traces.iscell[:, 1]


def show_roi(roi, seconds):
    shown = slice(0, min(f.shape[1], int(seconds * traces.fs)))
    fig = plt.figure(figsize=(15, 10), constrained_layout=True)
    grid = fig.add_gridspec(4, 2, width_ratios=[3, 1.3])
    trace_axes = [fig.add_subplot(grid[i, 0]) for i in range(3)]
    plots.plot_trace_audit(f, fneu, green, roi, traces.fs, shown, axes=trace_axes,
                           time_label=time_label)
    if ratio is not None:
        plots.plot_dff_trace(ratio, roi, traces.fs, shown, ax=fig.add_subplot(grid[3, 0]),
                             label="Δ(G/R)/(G/R)0", color="black", time_label=time_label)
    plots.plot_cell_fov(roi, mean_img, traces.labels,
                        axes=[fig.add_subplot(grid[0:2, 1]), fig.add_subplot(grid[2:4, 1])])
    fig.suptitle(f"ROI {roi}: classifier p(cell) {p_cell[roi]:.2f}, "
                 f"median F0 {table['median_f0'][roi]:.1f}, "
                 f"F0 invalid in {table['frac_invalid_f0'][roi]:.0%} of frames")
    plt.show()


roi_slider = widgets.IntSlider(value=0, min=0, max=f.shape[0] - 1, description="ROI",
                               continuous_update=False)
seconds_slider = widgets.IntSlider(value=300, min=10, max=max(10, int(f.shape[1] / traces.fs)),
                                   step=10, description="Seconds", continuous_update=False)
display(widgets.HBox([roi_slider, seconds_slider]),
        widgets.interactive_output(show_roi, {"roi": roi_slider, "seconds": seconds_slider}))